In [ ]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import pandas as pd
from pathlib import Path
import json
import os

class AirlinePriceApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Airline Price Calculator")
        self.root.geometry("1000x700")
        
        # Modern gradient background colors
        self.root.configure(bg="#0f172a")
        
        self.data = None
        self.current_results = []
        self.config_file = "airline_calculator_config.json"
        self.weight_columns = {
            "Minimum": "Minimum",
            "100-299": "100-299",
            "300-499": "300-499",
            "500-999": "500-999",
            ">999": ">999"
        }
        
        self.setup_ui()
        self.load_last_file()
        
    def setup_ui(self):
        # Main container with padding
        main = tk.Frame(self.root, bg="#0f172a")
        main.pack(fill=tk.BOTH, expand=True, padx=40, pady=30)
        
        # Header with modern styling
        header_frame = tk.Frame(main, bg="#0f172a")
        header_frame.pack(fill=tk.X, pady=(0, 30))
        
        title = tk.Label(header_frame, text="Airline Price Calculator", 
                        font=("SF Pro Display", 32, "bold"), 
                        bg="#0f172a", fg="#ffffff")
        title.pack(anchor=tk.W)
        
        subtitle = tk.Label(header_frame, text="Calculate shipping rates with dynamic markup", 
                           font=("SF Pro Display", 13), 
                           bg="#0f172a", fg="#94a3b8")
        subtitle.pack(anchor=tk.W, pady=(5, 0))
        
        # Load data section - modern card style
        load_card = tk.Frame(main, bg="#1e293b", highlightthickness=1, 
                            highlightbackground="#334155")
        load_card.pack(fill=tk.X, pady=(0, 20))
        
        load_inner = tk.Frame(load_card, bg="#1e293b")
        load_inner.pack(fill=tk.X, padx=25, pady=20)
        
        self.load_btn = tk.Button(load_inner, text="📁  Load Data File", 
                                  command=self.load_data,
                                  font=("SF Pro Display", 11, "bold"), 
                                  bg="#3b82f6", fg="white",
                                  cursor="hand2", relief=tk.FLAT,
                                  padx=25, pady=12,
                                  activebackground="#2563eb",
                                  borderwidth=0)
        self.load_btn.pack(side=tk.LEFT)
        
        self.status_label = tk.Label(load_inner, text="No data loaded", 
                                     font=("SF Pro Display", 11),
                                     bg="#1e293b", fg="#94a3b8")
        self.status_label.pack(side=tk.LEFT, padx=20)
        
        # Input section - glass morphism style
        input_card = tk.Frame(main, bg="#1e293b", highlightthickness=1, 
                             highlightbackground="#334155")
        input_card.pack(fill=tk.X, pady=(0, 20))
        
        input_inner = tk.Frame(input_card, bg="#1e293b")
        input_inner.pack(fill=tk.BOTH, padx=30, pady=30)
        
        # Grid layout for inputs
        input_grid = tk.Frame(input_inner, bg="#1e293b")
        input_grid.pack(fill=tk.X)
        
        # Configure grid columns for responsive layout
        input_grid.columnconfigure(0, weight=1)
        input_grid.columnconfigure(1, weight=1)
        
        # POL
        pol_frame = self.create_input_field(input_grid, "Port of Loading", "JFK, LAX, ORD...")
        pol_frame.grid(row=0, column=0, sticky=tk.EW, padx=(0, 15), pady=(0, 20))
        self.pol_entry = pol_frame.winfo_children()[1]
        
        # POD
        pod_frame = self.create_input_field(input_grid, "Port of Discharge", "LHR, NRT, FRA...")
        pod_frame.grid(row=0, column=1, sticky=tk.EW, padx=(15, 0), pady=(0, 20))
        self.pod_entry = pod_frame.winfo_children()[1]
        
        # Weight
        weight_frame = self.create_input_field(input_grid, "Weight (kg)", "Enter weight...")
        weight_frame.grid(row=1, column=0, sticky=tk.EW, padx=(0, 15), pady=(0, 20))
        self.weight_entry = weight_frame.winfo_children()[1]
        
        # Markup
        markup_frame = self.create_input_field(input_grid, "Markup (%)", "2")
        markup_frame.grid(row=1, column=1, sticky=tk.EW, padx=(15, 0), pady=(0, 20))
        self.markup_entry = markup_frame.winfo_children()[1]
        self.markup_entry.insert(0, "2")
        
        # Buttons
        button_frame = tk.Frame(input_inner, bg="#1e293b")
        button_frame.pack(fill=tk.X, pady=(10, 0))
        
        search_btn = tk.Button(button_frame, text="🔍  Search Rates", 
                              command=self.search_rates,
                              font=("SF Pro Display", 12, "bold"), 
                              bg="#10b981", fg="white",
                              cursor="hand2", relief=tk.FLAT,
                              padx=35, pady=14,
                              activebackground="#059669",
                              borderwidth=0)
        search_btn.pack(side=tk.LEFT, padx=(0, 15))
        
        export_btn = tk.Button(button_frame, text="💾  Export Results", 
                              command=self.export_results,
                              font=("SF Pro Display", 12, "bold"), 
                              bg="#6366f1", fg="white",
                              cursor="hand2", relief=tk.FLAT,
                              padx=35, pady=14,
                              activebackground="#4f46e5",
                              borderwidth=0)
        export_btn.pack(side=tk.LEFT)
        
        # Results section
        results_card = tk.Frame(main, bg="#1e293b", highlightthickness=1, 
                               highlightbackground="#334155")
        results_card.pack(fill=tk.BOTH, expand=True)
        
        results_inner = tk.Frame(results_card, bg="#1e293b")
        results_inner.pack(fill=tk.BOTH, expand=True, padx=30, pady=25)
        
        results_header = tk.Label(results_inner, text="Results", 
                                 font=("SF Pro Display", 18, "bold"),
                                 bg="#1e293b", fg="#ffffff")
        results_header.pack(anchor=tk.W, pady=(0, 15))
        
        # Modern treeview styling
        style = ttk.Style()
        style.theme_use('clam')
        
        style.configure("Modern.Treeview",
                       background="#0f172a",
                       foreground="#e2e8f0",
                       fieldbackground="#0f172a",
                       borderwidth=0,
                       font=("SF Pro Display", 11),
                       rowheight=40)
        
        style.configure("Modern.Treeview.Heading",
                       background="#1e293b",
                       foreground="#94a3b8",
                       borderwidth=0,
                       font=("SF Pro Display", 11, "bold"),
                       relief=tk.FLAT)
        
        style.map("Modern.Treeview",
                 background=[('selected', '#3b82f6')])
        
        style.map("Modern.Treeview.Heading",
                 background=[('active', '#1e293b')])
        
        # Treeview container
        tree_container = tk.Frame(results_inner, bg="#0f172a", 
                                 highlightthickness=1, highlightbackground="#334155")
        tree_container.pack(fill=tk.BOTH, expand=True)
        
        # Scrollbar
        scrollbar = ttk.Scrollbar(tree_container)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        self.tree = ttk.Treeview(tree_container, 
                                columns=("Airline", "Base Rate/kg", "Base Rate", "Sell Rate/kg", "Sell Rate"),
                                show="headings", 
                                yscrollcommand=scrollbar.set,
                                style="Modern.Treeview",
                                selectmode='browse')
        self.tree.pack(fill=tk.BOTH, expand=True, padx=2, pady=2)
        scrollbar.config(command=self.tree.yview)
        
        # Configure columns
        self.tree.heading("Airline", text="AIRLINE NAME")
        self.tree.heading("Base Rate/kg", text="BASE PER KG")
        self.tree.heading("Base Rate", text="TOTAL BASE")
        self.tree.heading("Sell Rate/kg", text="SELL PER KG")
        self.tree.heading("Sell Rate", text="TOTAL SELL")
        
        self.tree.column("Airline", width=280, anchor=tk.W)
        self.tree.column("Base Rate/kg", width=150, anchor=tk.CENTER)
        self.tree.column("Base Rate", width=150, anchor=tk.CENTER)
        self.tree.column("Sell Rate/kg", width=150, anchor=tk.CENTER)
        self.tree.column("Sell Rate", width=150, anchor=tk.CENTER)
        
    def create_input_field(self, parent, label_text, placeholder):
        """Create a modern input field with label"""
        frame = tk.Frame(parent, bg="#1e293b")
        
        label = tk.Label(frame, text=label_text, 
                        font=("SF Pro Display", 11, "bold"),
                        bg="#1e293b", fg="#94a3b8")
        label.pack(anchor=tk.W, pady=(0, 8))
        
        entry = tk.Entry(frame, 
                        font=("SF Pro Display", 12),
                        bg="#0f172a", fg="#e2e8f0",
                        relief=tk.FLAT,
                        insertbackground="#3b82f6",
                        highlightthickness=1,
                        highlightbackground="#334155",
                        highlightcolor="#3b82f6",
                        borderwidth=8)
        entry.pack(fill=tk.X)
        
        # Placeholder functionality
        entry.insert(0, placeholder)
        entry.config(fg="#475569")
        
        def on_focus_in(e):
            if entry.get() == placeholder:
                entry.delete(0, tk.END)
                entry.config(fg="#e2e8f0")
        
        def on_focus_out(e):
            if entry.get() == "":
                entry.insert(0, placeholder)
                entry.config(fg="#475569")
        
        entry.bind("<FocusIn>", on_focus_in)
        entry.bind("<FocusOut>", on_focus_out)
        
        return frame
    
    def load_data(self):
        file_path = filedialog.askopenfilename(
            title="Select Data File",
            filetypes=[("Excel files", "*.xlsx"), ("CSV files", "*.csv")]
        )
        
        if file_path:
            self.load_file(file_path)
    
    def load_file(self, file_path):
        """Load data from a specific file path"""
        try:
            if file_path.endswith('.xlsx'):
                self.data = pd.read_excel(file_path)
            else:
                self.data = pd.read_csv(file_path)
            
            self.status_label.config(text=f"✓ Loaded: {Path(file_path).name}", 
                                    fg="#10b981")
            
            self.save_last_file(file_path)
            
            messagebox.showinfo("Success", f"Data loaded successfully!\n\n{len(self.data)} records loaded from:\n{Path(file_path).name}")
        except FileNotFoundError:
            self.status_label.config(text="File not found", fg="#ef4444")
            if hasattr(self, 'attempting_auto_load'):
                pass
            else:
                messagebox.showerror("Error", f"File not found:\n{file_path}")
        except Exception as e:
            self.status_label.config(text="Error loading file", fg="#ef4444")
            messagebox.showerror("Error", f"Failed to load file:\n{str(e)}")
    
    def save_last_file(self, file_path):
        """Save the last used file path to config"""
        try:
            config = {"last_file": file_path}
            with open(self.config_file, 'w') as f:
                json.dump(config, f)
        except Exception as e:
            print(f"Could not save config: {e}")
    
    def load_last_file(self):
        """Load the last used file automatically on startup"""
        try:
            if os.path.exists(self.config_file):
                with open(self.config_file, 'r') as f:
                    config = json.load(f)
                    last_file = config.get("last_file")
                    
                    if last_file and os.path.exists(last_file):
                        self.attempting_auto_load = True
                        self.load_file(last_file)
                        delattr(self, 'attempting_auto_load')
                        self.status_label.config(
                            text=f"✓ Auto-loaded: {Path(last_file).name}", 
                            fg="#10b981"
                        )
                    else:
                        self.status_label.config(text="No data loaded", fg="#94a3b8")
        except Exception as e:
            print(f"Could not load last file: {e}")
            self.status_label.config(text="No data loaded", fg="#94a3b8")
    
    def get_weight_category(self, weight):
        """Determine weight category based on input weight"""
        try:
            w = float(weight)
            if w < 100:
                return "Minimum"
            elif 100 <= w < 300:
                return "100-299"
            elif 300 <= w < 500:
                return "300-499"
            elif 500 <= w < 1000:
                return "500-999"
            else:
                return ">999"
        except ValueError:
            return None
    
    def search_rates(self):
        # Clear previous results
        for item in self.tree.get_children():
            self.tree.delete(item)
        
        self.current_results = []
        
        if self.data is None:
            messagebox.showwarning("No Data", "Please load a data file first!")
            return
        
        # Get values and handle placeholders
        pol = self.pol_entry.get().strip()
        pod = self.pod_entry.get().strip()
        weight = self.weight_entry.get().strip()
        markup_str = self.markup_entry.get().strip()
        
        # Check for placeholder text
        if pol in ["JFK, LAX, ORD...", ""]:
            messagebox.showwarning("Missing Input", "Please enter Port of Loading!")
            return
        if pod in ["LHR, NRT, FRA...", ""]:
            messagebox.showwarning("Missing Input", "Please enter Port of Discharge!")
            return
        if weight in ["Enter weight...", ""]:
            messagebox.showwarning("Missing Input", "Please enter Weight!")
            return
        if markup_str == "":
            messagebox.showwarning("Missing Input", "Please enter Markup percentage!")
            return
        
        # Validate markup percentage
        try:
            markup = float(markup_str)
            if markup < 0:
                messagebox.showerror("Invalid Markup", "Markup percentage cannot be negative!")
                return
        except ValueError:
            messagebox.showerror("Invalid Markup", "Please enter a valid numeric markup percentage!")
            return
        
        weight_category = self.get_weight_category(weight)
        if weight_category is None:
            messagebox.showerror("Invalid Weight", "Please enter a valid numeric weight!")
            return
        
        # Filter data
        filtered = self.data[
            (self.data['POL'].str.upper() == pol.upper()) & 
            (self.data['POD'].str.upper() == pod.upper())
        ]
        
        if filtered.empty:
            messagebox.showinfo("No Results", 
                              f"No airlines found for route {pol} → {pod}")
            return
        
        # Display results and store them
        results_found = False
        for _, row in filtered.iterrows():
            if weight_category in row and pd.notna(row[weight_category]):
                rate_value = float(row[weight_category])
                weight_value = float(weight)
                
                # Check if this is Minimum rate (flat rate, not per kg)
                if weight_category == "Minimum":
                    # Minimum is a flat rate up to 100 kg, not multiplied
                    base_rate = rate_value
                    rate_display = "Flat Rate"
                    base_per_kg = base_rate / weight_value
                else:
                    # Regular per-kg calculation
                    base_rate = rate_value * weight_value
                    rate_display = f"${rate_value:.2f}"
                    base_per_kg = rate_value
                
                # Calculate sell rate with markup
                sell_rate = base_rate * (1 + markup / 100)
                sell_per_kg = sell_rate / weight_value
                
                # Store result
                result_data = {
                    'Airline Name': row['Airline Name'],
                    'POL': pol.upper(),
                    'POD': pod.upper(),
                    'Weight (kg)': weight_value,
                    'Weight Category': weight_category,
                    'Rate per kg': rate_value if weight_category != "Minimum" else "N/A (Flat Rate)",
                    'Base Rate (Total)': base_rate,
                    'Base Rate per kg': base_per_kg,
                    'Markup (%)': markup,
                    'Sell Rate (Total)': sell_rate,
                    'Sell Rate per kg': sell_per_kg
                }
                self.current_results.append(result_data)
                
                # Display in tree
                self.tree.insert("", tk.END, values=(
                    row['Airline Name'],
                    f"${base_per_kg:.2f}",
                    f"${base_rate:.2f}",
                    f"${sell_per_kg:.2f}",
                    f"${sell_rate:.2f}"
                ))
                results_found = True
        
        if not results_found:
            messagebox.showinfo("No Results", 
                              f"No rates found for weight category: {weight_category} kg")
    
    def export_results(self):
        """Export current search results to Excel or CSV"""
        if not self.current_results:
            messagebox.showwarning("No Results", 
                                 "Please perform a search first before exporting!")
            return
        
        file_path = filedialog.asksaveasfilename(
            title="Save Results As",
            defaultextension=".xlsx",
            filetypes=[("Excel files", "*.xlsx"), ("CSV files", "*.csv")]
        )
        
        if file_path:
            try:
                df = pd.DataFrame(self.current_results)
                
                if file_path.endswith('.xlsx'):
                    df.to_excel(file_path, index=False, sheet_name='Search Results')
                else:
                    df.to_csv(file_path, index=False)
                
                messagebox.showinfo("Success", 
                                  f"Results exported successfully!\n\n{len(self.current_results)} records saved to:\n{Path(file_path).name}")
            except Exception as e:
                messagebox.showerror("Export Error", 
                                   f"Failed to export results:\n{str(e)}")

if __name__ == "__main__":
    root = tk.Tk()
    app = AirlinePriceApp(root)
    root.mainloop()